<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/premi-secundaria/++theme++genweb6.theme/img/logo.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización y robotización de procesos</p>
  </div>

</div>

# 02 · Detectar procesos automatizables

## Laboratorio: justificar una oportunidad antes de implementarla

Caso conductor: observatorio territorial de deuda tributaria publicada. **Ocho procesos y datos completamente sintéticos.** No se realizan descargas, tratamientos de datos personales reales, envíos ni cambios de archivos.

El objetivo es explicar por qué probar una oportunidad y qué evidencias podrían cambiar esa recomendación.

**Versión del alumnado.** Parte guiada completa y ejercicios pendientes.

## Objetivos y alcance

Delimitar un proceso, medir su carga, incorporar revisión y excepciones, separar filtros de puntuaciones y contrastar sensibilidad. El resultado será una cartera con estados y una propuesta de piloto.

No implementamos operaciones con pathlib, Excel, PDFs, APIs ni modelos de IA: corresponden a temas posteriores. El Python de esta sesión sirve para calcular escenarios sobre fichas en memoria.

## Cómo trabajar

Ejecuta la guía en orden y anticipa el resultado antes de cada cálculo. Completa los doce ejercicios, que dependen de la guía y no de otras respuestas. Activa cada bandera `completado_ejXX` para comprobar tu solución. Una bandera False significa **control omitido**, no tarea aprobada.

Reinicia el kernel y ejecuta todo al terminar. Los textos y las decisiones metodológicas requieren revisión humana aunque sus controles formales pasen.

## Entorno

Python 3.10 o superior. La práctica usa solo biblioteca estándar: math, statistics, copy, collections y pprint. Para abrir el archivo necesitas un editor compatible con Jupyter. Los scripts `.py` no requieren Jupyter ni paquetes adicionales.

Los gráficos estáticos adjuntos se muestran desde `recursos/`. Si mueves el notebook, conserva esas rutas. No es necesario ejecutar Matplotlib.

In [ ]:
"""Modelo docente: criterios ilustrativos, no una norma ni autorización de ejecución."""
from math import isfinite, isclose
from statistics import mean, median
from copy import deepcopy
from collections import Counter
from pprint import pprint

FACTORES = ("volumen", "estabilidad", "estructura", "calidad", "impacto", "facilidad")
PESOS = {"volumen": .20, "estabilidad": .20, "estructura": .15,
         "calidad": .15, "impacto": .20, "facilidad": .10}
VERSION_MODELO = "priorizacion-aula-1.0"




## 1. Un proceso con frontera

P02 comienza cuando recibimos registros estructurados y termina cuando entregamos aceptados, incidencias y un resumen revisable. Su unidad es **un registro**.

El proceso más amplio de preparar un informe incluye otras tareas. No podemos sumar el ahorro del informe y el de la validación si ya se ha contado el mismo trabajo en ambos.

### Ficha descriptiva del ejemplo

| Campo | Valor sintético |
|---|---|
| Objetivo | Aplicar reglas acordadas con evidencia por registro. |
| Inicio | Datos estructurados disponibles. |
| Final | Resumen e incidencias preparados para revisión. |
| Responsable | Equipo de análisis. |
| Unidad | Registro. |
| Dependencia | Fuente y contrato de entrada contrastados. |

Esta ficha metodológica amplía la estructura numérica de la cartera. En un proyecto real guardaríamos además procedencia, fecha, confianza y responsable de cada estimación.

## 2. Medir una muestra sin ocultar su variación

La media incorpora la carga de casos largos. La mediana describe el centro de la muestra. Conservamos ambos datos y la definición de esfuerzo observado. Los siguientes valores ilustran minutos por caso, no son medidas del proyecto real.

In [ ]:
tiempos_observados = [5, 5, 6, 6, 7, 7, 8, 20]
print("n:", len(tiempos_observados))
print("Media:", mean(tiempos_observados))
print("Mediana:", median(tiempos_observados))
print("Mínimo y máximo:", min(tiempos_observados), max(tiempos_observados))


### Interpretación

La media es 8 minutos y la mediana 6,5. El valor 20 puede corresponder a una excepción real. No se elimina por resultar incómodo. Ocho observaciones no garantizan representar cierres, cambios de formato o casos poco frecuentes.

Distingue **esfuerzo humano** de espera y de tiempo de ciclo. Varias personas pueden trabajar en paralelo, por lo que persona-minutos y minutos de reloj son medidas diferentes.

## 3. La cartera del caso

Cada ficha declara unidad, volumen anual y minutos por esa misma unidad. `min_operacion` es intervención **humana** operativa en el proceso futuro, no tiempo de máquina. `min_revision` es revisión humana habitual. `min_excepcion` es trabajo humano adicional cuando una unidad tiene excepción.

Los minutos manuales actuales ya incluyen la mezcla habitual de casos. No sumamos otra vez sus excepciones al esfuerzo actual. Los valores futuros son supuestos por contrastar.

In [ ]:
cartera = [{'id': 'P01', 'nombre': 'Organizar evidencias', 'volumen_anual': 2400, 'min_manual': 1.8, 'min_revision': 0.1, 'p_excepcion': 0.05, 'min_excepcion': 3, 'horas_impl': 12, 'horas_mant': 6, 'factores': {'volumen': 4, 'estabilidad': 5, 'estructura': 5, 'calidad': 4, 'impacto': 3, 'facilidad': 5}, 'riesgo': 2, 'autorizacion': True, 'responsable': 'Equipo de análisis', 'unidad': 'archivo', 'min_operacion': 0.1}, {'id': 'P02', 'nombre': 'Validar registros', 'volumen_anual': 12000, 'min_manual': 0.8, 'min_revision': 0.03, 'p_excepcion': 0.08, 'min_excepcion': 2, 'horas_impl': 28, 'horas_mant': 12, 'factores': {'volumen': 5, 'estabilidad': 5, 'estructura': 5, 'calidad': 4, 'impacto': 5, 'facilidad': 4}, 'riesgo': 2, 'autorizacion': True, 'responsable': 'Equipo de análisis', 'unidad': 'registro', 'min_operacion': 0.02}, {'id': 'P03', 'nombre': 'Preparar informes', 'volumen_anual': 48, 'min_manual': 90, 'min_revision': 15, 'p_excepcion': 0.1, 'min_excepcion': 30, 'horas_impl': 20, 'horas_mant': 10, 'factores': {'volumen': 3, 'estabilidad': 4, 'estructura': 5, 'calidad': 4, 'impacto': 4, 'facilidad': 4}, 'riesgo': 2, 'autorizacion': True, 'responsable': 'Equipo de análisis', 'unidad': 'informe', 'min_operacion': 3}, {'id': 'P04', 'nombre': 'Resolver territorio', 'volumen_anual': 6000, 'min_manual': 4, 'min_revision': 0.5, 'p_excepcion': 0.35, 'min_excepcion': 8, 'horas_impl': 80, 'horas_mant': 24, 'factores': {'volumen': 4, 'estabilidad': 2, 'estructura': 3, 'calidad': 2, 'impacto': 5, 'facilidad': 2}, 'riesgo': 4, 'autorizacion': None, 'responsable': 'Equipo de análisis', 'unidad': 'registro', 'min_operacion': 0.2}, {'id': 'P05', 'nombre': 'Preparar correos', 'volumen_anual': 52, 'min_manual': 20, 'min_revision': 5, 'p_excepcion': 0.05, 'min_excepcion': 10, 'horas_impl': 8, 'horas_mant': 4, 'factores': {'volumen': 2, 'estabilidad': 4, 'estructura': 5, 'calidad': 5, 'impacto': 3, 'facilidad': 4}, 'riesgo': 3, 'autorizacion': True, 'responsable': 'Equipo de análisis', 'unidad': 'correo', 'min_operacion': 1}, {'id': 'P06', 'nombre': 'Publicar datos nominales', 'volumen_anual': 12, 'min_manual': 60, 'min_revision': 10, 'p_excepcion': 0.1, 'min_excepcion': 10, 'horas_impl': 40, 'horas_mant': 20, 'factores': {'volumen': 2, 'estabilidad': 4, 'estructura': 5, 'calidad': 4, 'impacto': 5, 'facilidad': 3}, 'riesgo': 5, 'autorizacion': False, 'responsable': 'Equipo de análisis', 'unidad': 'publicación', 'min_operacion': 2}, {'id': 'P07', 'nombre': 'Interpretar normativa', 'volumen_anual': 24, 'min_manual': 120, 'min_revision': 90, 'p_excepcion': 0.4, 'min_excepcion': 60, 'horas_impl': 60, 'horas_mant': 20, 'factores': {'volumen': 2, 'estabilidad': 1, 'estructura': 2, 'calidad': 3, 'impacto': 5, 'facilidad': 1}, 'riesgo': 4, 'autorizacion': True, 'responsable': 'Equipo de análisis', 'unidad': 'consulta', 'min_operacion': 10}, {'id': 'P08', 'nombre': 'Contrastar fuentes', 'volumen_anual': 120, 'min_manual': 12, 'min_revision': 2, 'p_excepcion': 0.1, 'min_excepcion': 8, 'horas_impl': 18, 'horas_mant': 8, 'factores': {'volumen': 3, 'estabilidad': 3, 'estructura': 4, 'calidad': None, 'impacto': 4, 'facilidad': 3}, 'riesgo': None, 'autorizacion': True, 'responsable': None, 'unidad': 'contraste', 'min_operacion': 1}]


In [ ]:
for f in cartera:
    print(f["id"], f["nombre"], "·", f["volumen_anual"], f["unidad"] + "/año")
p02 = deepcopy(next(f for f in cartera if f["id"] == "P02"))


## 4. Condiciones de entrada del modelo

Los tiempos deben ser números finitos no negativos. El volumen es un entero no negativo y las proporciones se expresan de cero a uno. Rechazamos booleanos como cantidades. La valoración de factores y riesgo es un entero de uno a cinco o None si es desconocida.

Autorización permite True, False o None. Un responsable desconocido se declara None. No se adivinan datos ausentes ni se convierten cadenas numéricas automáticamente.

In [ ]:
def es_numero(valor):
    return type(valor) in (int, float) and isfinite(valor)

def validar_pesos(pesos):
    if not isinstance(pesos, dict) or set(pesos) != set(FACTORES):
        raise ValueError("Los pesos deben contener exactamente los seis factores")
    if any(not es_numero(x) or x < 0 for x in pesos.values()):
        raise ValueError("Los pesos deben ser finitos y no negativos")
    if not isclose(sum(pesos.values()), 1.0, abs_tol=1e-9):
        raise ValueError("Los pesos deben sumar uno")


### Por qué validamos pesos

El conjunto de factores debe coincidir exactamente con el acordado. Los pesos son finitos y no negativos y deben sumar uno dentro de una tolerancia numérica pequeña. Cambiar de 0,20 a 20 es un cambio de escala, no una modificación menor.

In [ ]:
def validar_ficha(f):
    if not isinstance(f, dict):
        raise ValueError("La ficha debe ser un diccionario")
    requeridos = {"id", "nombre", "unidad", "volumen_anual", "min_manual", "min_operacion",
                  "min_revision", "p_excepcion", "min_excepcion", "horas_impl",
                  "horas_mant", "factores", "riesgo", "autorizacion", "responsable"}
    if not requeridos.issubset(f):
        raise ValueError("Faltan campos obligatorios en la ficha")
    if any(not isinstance(f[k], str) or not f[k].strip() for k in ("id", "nombre", "unidad")):
        raise ValueError("ID, nombre y unidad deben ser textos no vacíos")
    if type(f["volumen_anual"]) is not int or f["volumen_anual"] < 0:
        raise ValueError("El volumen anual debe ser un entero no negativo")
    tiempos = ("min_manual", "min_operacion", "min_revision", "min_excepcion",
               "horas_impl", "horas_mant")
    if any(not es_numero(f[k]) or f[k] < 0 for k in tiempos):
        raise ValueError("Los tiempos deben ser finitos y no negativos")
    if not es_numero(f["p_excepcion"]) or not 0 <= f["p_excepcion"] <= 1:
        raise ValueError("La proporción de excepciones debe estar entre cero y uno")
    if not isinstance(f["factores"], dict) or set(f["factores"]) != set(FACTORES):
        raise ValueError("La ficha debe declarar los seis factores")
    for valor in list(f["factores"].values()) + [f["riesgo"]]:
        if valor is not None and (type(valor) is not int or not 1 <= valor <= 5):
            raise ValueError("Las valoraciones deben ser enteros 1–5 o None")
    if f["autorizacion"] is not None and type(f["autorizacion"]) is not bool:
        raise ValueError("Autorización debe ser True, False o None")
    if f["responsable"] is not None and (
        not isinstance(f["responsable"], str) or not f["responsable"].strip()
    ):
        raise ValueError("Responsable debe ser texto no vacío o None")


In [ ]:
validar_ficha(p02)
validar_pesos(PESOS)
print("Ficha P02 y pesos compatibles con el contrato")


## 5. Modelo de esfuerzo

$ t_f = t_o + t_r + p_e t_e $

$ H_m = V t_m/60 $, $H_f = V t_f/60$. El esfuerzo bruto liberado es $H_m-H_f$.

Primer año: descontamos implantación y mantenimiento. En un año recurrente descontamos solo mantenimiento, suponiendo que no hay otra implantación. La recuperación en meses reparte uniformemente volumen y mantenimiento; es recuperación de **esfuerzo**, no rentabilidad financiera.

In [ ]:
def calcular_metricas(f):
    validar_ficha(f)
    minutos_futuros = f["min_operacion"] + f["min_revision"] + f["p_excepcion"] * f["min_excepcion"]
    manual = f["volumen_anual"] * f["min_manual"] / 60
    futuro = f["volumen_anual"] * minutos_futuros / 60
    ahorro_bruto = manual - futuro
    ahorro_recurrente = ahorro_bruto - f["horas_mant"]
    primer_anio = ahorro_recurrente - f["horas_impl"]
    retorno_meses = (12 * f["horas_impl"] / ahorro_recurrente
                     if ahorro_recurrente > 0 else None)
    return {"min_futuros": minutos_futuros, "horas_manual": manual,
            "horas_futuras": futuro, "horas_liberadas_brutas": ahorro_bruto,
            "horas_netas_recurrentes": ahorro_recurrente,
            "horas_netas_primer_anio": primer_anio,
            "retorno_meses": retorno_meses}


In [ ]:
metricas_p02 = calcular_metricas(p02)
for k, v in metricas_p02.items():
    print(k, ":", None if v is None else round(v, 4))
assert isclose(metricas_p02["min_futuros"], 0.21)
assert isclose(metricas_p02["horas_manual"], 160)
assert isclose(metricas_p02["horas_futuras"], 42)
assert isclose(metricas_p02["horas_netas_primer_anio"], 78)


### Lectura del resultado

La diferencia por registro es 0,59 minutos y libera 118 horas brutas al año. Restar 28 horas iniciales y 12 de mantenimiento produce 78 horas netas en el primer año. La capacidad recurrente es 106 horas/año y la recuperación ilustrativa de esfuerzo es de unos 3,17 meses.

Si el trabajo no reduce un gasto evitable, no debe describirse como ahorro de caja. Licencias e infraestructura requerirían un análisis adicional en euros.

## 6. Puntuación transparente

Seis factores: volumen, estabilidad, estructura, calidad, impacto y facilidad. Los pesos son preferencias docentes, no una norma externa. Cada valoración necesita anclajes y evidencia. Los niveles se usan de forma compensatoria con fines ilustrativos: no son probabilidades ni intervalos iguales validados empíricamente.

$ S=100\sum_j w_j(s_j-1)/4 $. Con unos en todos los factores se obtiene cero; con cincos, cien. Un dato desconocido produce None y obliga a recoger evidencia.

In [ ]:
def puntuar(factores, pesos):
    validar_pesos(pesos)
    if not isinstance(factores, dict) or set(factores) != set(FACTORES):
        raise ValueError("Se requieren exactamente los seis factores")
    for valor in factores.values():
        if valor is not None and (type(valor) is not int or not 1 <= valor <= 5):
            raise ValueError("Los factores son enteros 1–5 o None")
    if any(v is None for v in factores.values()):
        return None  # Desconocido no equivale a cero.
    return 100 * sum(pesos[k] * (factores[k] - 1) / 4 for k in FACTORES)


In [ ]:
score_p02 = puntuar(p02["factores"], PESOS)
print("P02:", score_p02)
faltan_datos = {**p02["factores"], "calidad": None}
print("Sin evidencia de calidad:", puntuar(faltan_datos, PESOS))
assert isclose(score_p02, 93.75)


## 7. Los filtros tienen prioridad

Autorización False bloquea. Información esencial desconocida deja pendiente. Riesgo 4–5 obliga a rediseñar en esta política de aula. Solo con la ficha completa consideramos puntuación al menos 70 y balance positivo del primer año como condiciones para ser candidato a piloto.

El criterio es deliberadamente explícito y discutible. No equivale a una autorización normativa ni obliga a descartar mejoras cuyo valor principal sea calidad o servicio.

In [ ]:
def evaluar(f, pesos=None):
    if pesos is None:
        pesos = PESOS
    metricas = calcular_metricas(f)
    score = puntuar(f["factores"], pesos)
    if f["autorizacion"] is False:
        estado, motivo = "bloqueado", "Falta autorización: no se ejecuta el proceso"
    elif any(f[k] is None for k in ("autorizacion", "responsable", "riesgo")) or score is None:
        estado, motivo = "pendiente", "Faltan evidencias o una persona responsable"
    elif f["riesgo"] >= 4:
        estado, motivo = "redisenar", "Riesgo alto: revisar alcance y controles"
    elif score >= 70 and metricas["horas_netas_primer_anio"] > 0:
        estado, motivo = "candidato_piloto", "Supera los criterios docentes; requiere validar un piloto"
    else:
        estado, motivo = "analizar", "Revisar valor, alcance o supuestos antes de priorizar"
    return {"id": f["id"], "nombre": f["nombre"], "puntuacion": score,
            "estado": estado, "motivo": motivo, "metricas": metricas,
            "version_modelo": VERSION_MODELO}


In [ ]:
for identificador in ["P02", "P04", "P06", "P07", "P08"]:
    f = next(x for x in cartera if x["id"] == identificador)
    r = evaluar(f)
    print(identificador, r["estado"], "·", r["motivo"])


## 8. Cartera completa y orden estable

Se valida cada ficha y se rechazan identificadores repetidos en vez de elegir una versión silenciosamente. Los candidatos se ordenan por puntuación. Si empatan, el ID estabiliza técnicamente el orden; el desempate de negocio exige deliberación, no una preferencia alfabética.

In [ ]:
def evaluar_cartera(fichas, pesos=None):
    if not isinstance(fichas, list):
        raise ValueError("La cartera debe ser una lista")
    ids = set()
    for f in fichas:
        validar_ficha(f)
        if f["id"] in ids:
            raise ValueError("ID repetido en la cartera")
        ids.add(f["id"])
    if pesos is None:
        pesos = PESOS
    validar_pesos(pesos)
    return [evaluar(f, pesos) for f in fichas]

def ordenar_candidatos(resultados):
    candidatos = [r for r in resultados if r["estado"] == "candidato_piloto"]
    # Un empate usa ID para estabilidad técnica, no como preferencia de negocio.
    return sorted(candidatos, key=lambda r: (-r["puntuacion"], r["id"]))


In [ ]:
resultados = evaluar_cartera(cartera)
for r in resultados:
    print(r["id"], r["puntuacion"], r["estado"],
          round(r["metricas"]["horas_netas_primer_anio"], 2))
prioridad = ordenar_candidatos(resultados)
print("Candidatos:", [r["id"] for r in prioridad])
assert [r["id"] for r in prioridad] == ["P02", "P01", "P03"]
assert len(resultados) == len(cartera)


### Representar el balance

![Balance de horas netas de ocho oportunidades ficticias](recursos/horas.png)

No sumamos automáticamente estas barras: varias tareas pueden compartir pasos o depender de la misma mejora. Un beneficio aislado no define un calendario de implantación.

## 9. Sensibilidad a excepciones

Variamos una proporción dejando los demás supuestos fijos. Este análisis muestra qué decisión cambiaría; no representa un intervalo de confianza ni calcula la probabilidad de un resultado.

In [ ]:
def simular_excepciones(f, proporciones):
    filas = []
    for proporcion in proporciones:
        escenario = deepcopy(f)
        escenario["p_excepcion"] = proporcion
        filas.append({"p_excepcion": proporcion,
                      **calcular_metricas(escenario)})
    return filas


In [ ]:
escenarios = simular_excepciones(p02, [0.02, 0.08, 0.30])
for fila in escenarios:
    print(fila["p_excepcion"], round(fila["horas_netas_primer_anio"], 2))
assert isclose(escenarios[-1]["horas_netas_primer_anio"], -10)


### Un punto crítico del escenario

Para volumen y esfuerzo adicional positivos, al despejar el balance del primer año obtenemos:

$ p_e^* = [t_m-t_o-t_r-60(H_i+H_{mant})/V]/t_e $.

En P02 es 0,275. No se trata de un umbral de errores permitidos: mide el equilibrio del modelo de esfuerzo con estos supuestos. Las excepciones pueden estar detectadas y correctamente resueltas.

In [ ]:
umbral_p02 = (p02["min_manual"] - p02["min_operacion"] - p02["min_revision"]
    - 60 * (p02["horas_impl"] + p02["horas_mant"]) / p02["volumen_anual"]
) / p02["min_excepcion"]
print("Proporción de equilibrio:", round(umbral_p02, 4))
assert isclose(umbral_p02, 0.275)


![Sensibilidad de P02 a la carga adicional](recursos/sensibilidad.png)

## 10. Sensibilidad a preferencias

Un equipo que prioriza impacto puede acordar otro reparto: 50 % para impacto y 10 % para cada factor restante. Recalculamos el conjunto sin cambiar las fichas ni ocultar los estados.

In [ ]:
pesos_impacto = {k: 0.10 for k in FACTORES}
pesos_impacto["impacto"] = 0.50
alternativa = ordenar_candidatos(evaluar_cartera(cartera, pesos_impacto))
print("Base:", [r["id"] for r in prioridad])
print("Impacto:", [r["id"] for r in alternativa])
assert [r["id"] for r in alternativa] == ["P02", "P03", "P01"]


### Qué nos dice el cambio

P02 permanece primero en esta comparación. P01 y P03 intercambian posiciones. La recomendación de cuál evaluar después depende de la preferencia declarada. La clasificación no es un hecho natural de los datos.

Revisa también si dos factores se solapan: ponderar varias veces el mismo atributo puede reforzar sin querer una preferencia.

## 11. Comprobaciones de integridad

El modelo no modifica la cartera. Ejecutar de nuevo con los mismos valores y pesos devuelve el mismo resultado funcional. Eso no comprueba que una futura integración externa sea idempotente ni que los supuestos sean correctos.

In [ ]:
antes = deepcopy(cartera)
a = evaluar_cartera(cartera)
b = evaluar_cartera(cartera)
assert a == b
assert cartera == antes
assert evaluar_cartera([]) == []
print("Entrada intacta, resultado repetible y cartera vacía admitida")


## 12. Propuesta de piloto

P02 parece una primera oportunidad para ensayar: puntuación 93,75 y 78 horas netas estimadas. La recomendación es **probar una hipótesis**, no afirmar una mejora demostrada.

El piloto necesita una referencia manual revisada, alcance pequeño, un responsable, métricas comparables y condiciones de detención. La estimación de excepciones es una incertidumbre que debe medirse. No se ejecutan acciones con datos reales desde esta práctica.

## Ejercicios

Completa cada tarea y justifica la interpretación. Las comprobaciones de texto solo evalúan forma: el docente debe revisar su contenido. En todas las tareas puedes usar las funciones de la parte guiada. Las variantes de solución son válidas si respetan el contrato y explican sus límites.

## Ejercicio 01 · Delimitar una oportunidad

Define `ficha_descriptiva` con objetivo, inicio, fin, unidad y responsable para P02. El inicio debe poder observarse y el final debe ser un resultado verificable.

**Pista:** Describe una tarea de validación, no todo el observatorio.

In [ ]:
ficha_descriptiva = {"objetivo":"", "inicio":"", "fin":"", "unidad":"", "responsable":""}

In [ ]:
completado_ej01 = False
if completado_ej01:
    assert set(ficha_descriptiva) == {"objetivo", "inicio", "fin", "unidad", "responsable"}
    assert all(isinstance(v,str) and v.strip() for v in ficha_descriptiva.values())
    print("Ejercicio 01: controles superados")
else:
    print("Ejercicio 01: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 02 · Interpretar una muestra

Calcula `media_ej2` y `mediana_ej2` de [5,5,6,6,7,7,8,20]. Escribe `interpretacion_ej2` explicando por qué no borrarías automáticamente el caso de 20 minutos.

**Pista:** Media y mediana responden a aspectos distintos de la distribución.

In [ ]:
muestra_ej2 = [5,5,6,6,7,7,8,20]
media_ej2 = None  # TODO
mediana_ej2 = None  # TODO
interpretacion_ej2 = ""

In [ ]:
completado_ej02 = False
if completado_ej02:
    assert isclose(media_ej2,8)
    assert isclose(mediana_ej2,6.5)
    assert len(interpretacion_ej2)>40
    print("Ejercicio 02: controles superados")
else:
    print("Ejercicio 02: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 03 · Carga manual con unidades coherentes

Define `carga_manual_ej3` con las horas anuales de P02 usando volumen y minutos manuales. Explica en `unidades_ej3` qué se cancela al dividir por 60.

**Pista:** 12.000 registros/año por 0,80 minutos/registro.

In [ ]:
carga_manual_ej3 = None  # TODO
unidades_ej3 = ""

In [ ]:
completado_ej03 = False
if completado_ej03:
    assert isclose(carga_manual_ej3,160)
    assert len(unidades_ej3)>30
    print("Ejercicio 03: controles superados")
else:
    print("Ejercicio 03: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 04 · Revisión y excepción en el proceso futuro

Implementa `esfuerzo_futuro(operacion, revision, p_exc, extra)` con los parámetros en minutos por unidad. Calcula `futuro_ej4` para P02.

**Pista:** El trabajo adicional solo se aplica a la proporción que tiene excepción.

In [ ]:
def esfuerzo_futuro(operacion, revision, p_exc, extra):
    return None  # TODO
futuro_ej4 = None  # TODO

In [ ]:
completado_ej04 = False
if completado_ej04:
    assert isclose(futuro_ej4,0.21)
    assert isclose(esfuerzo_futuro(1,2,0,10),3)
    assert isclose(esfuerzo_futuro(1,2,1,10),13)
    print("Ejercicio 04: controles superados")
else:
    print("Ejercicio 04: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 05 · Primer año frente a esfuerzo recurrente

Obtén `metricas_ej5` para P02. Guarda `primer_anio_ej5`, `recurrente_ej5` y `meses_ej5`. Escribe qué hipótesis permite expresar la recuperación en meses.

**Pista:** La implantación se descuenta una sola vez. El mantenimiento permanece.

In [ ]:
metricas_ej5 = calcular_metricas(p02)
primer_anio_ej5 = None  # TODO
recurrente_ej5 = None  # TODO
meses_ej5 = None  # TODO
hipotesis_ej5 = ""

In [ ]:
completado_ej05 = False
if completado_ej05:
    assert isclose(primer_anio_ej5,78)
    assert isclose(recurrente_ej5,106)
    assert isclose(meses_ej5,12*28/106)
    assert len(hipotesis_ej5)>30
    print("Ejercicio 05: controles superados")
else:
    print("Ejercicio 05: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 06 · Reconstruir la puntuación

Define `contribuciones_ej6` como un diccionario con la contribución de cada factor de P02 a la escala 0–100. Suma en `score_ej6`.

**Pista:** Cada factor aporta 100*peso*(valor-1)/4.

In [ ]:
contribuciones_ej6 = {}  # TODO
score_ej6 = None  # TODO

In [ ]:
completado_ej06 = False
if completado_ej06:
    assert set(contribuciones_ej6)==set(FACTORES)
    assert isclose(contribuciones_ej6["calidad"],11.25)
    assert isclose(score_ej6,93.75)
    print("Ejercicio 06: controles superados")
else:
    print("Ejercicio 06: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 07 · Un veto no se compensa con puntuación

Crea una copia de P02 con autorización False y todos los factores en 5. Evalúala como `veto_ej7`. ¿Debe pasar a piloto?

**Pista:** Usa deepcopy para no modificar la ficha que usan otros ejercicios.

In [ ]:
ficha_veto_ej7 = deepcopy(p02)
# TODO: modifica autorización y factores en la copia
veto_ej7 = None

In [ ]:
completado_ej07 = False
if completado_ej07:
    assert isclose(veto_ej7["puntuacion"],100)
    assert veto_ej7["estado"]=="bloqueado"
    assert p02["autorizacion"] is True
    print("Ejercicio 07: controles superados")
else:
    print("Ejercicio 07: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 08 · Conservar toda la cartera y ordenar solo candidatos

Evalúa la cartera, calcula `ids_ej8` con los candidatos ordenados y `estados_ej8` como frecuencias de estado.

**Pista:** Los bloqueados y pendientes no se borran del resultado completo.

In [ ]:
resultados_ej8 = evaluar_cartera(cartera)
ids_ej8 = []  # TODO
estados_ej8 = {}  # TODO

In [ ]:
completado_ej08 = False
if completado_ej08:
    assert ids_ej8==["P02","P01","P03"]
    assert sum(estados_ej8.values())==8
    assert estados_ej8=={"candidato_piloto":3,"pendiente":2,"analizar":1,"bloqueado":1,"redisenar":1}
    print("Ejercicio 08: controles superados")
else:
    print("Ejercicio 08: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 09 · Cambiar una hipótesis crítica

Simula P02 con proporciones 0,08 y 0,30 y guarda `balance_base_ej9` y `balance_alto_ej9`. Escribe `decision_ej9` explicando qué medirías en un piloto.

**Pista:** Varía solo la proporción para aislar su efecto.

In [ ]:
escenarios_ej9 = simular_excepciones(p02,[0.08,0.30])
balance_base_ej9 = None  # TODO
balance_alto_ej9 = None  # TODO
decision_ej9 = ""

In [ ]:
completado_ej09 = False
if completado_ej09:
    assert isclose(balance_base_ej9,78)
    assert isclose(balance_alto_ej9,-10)
    assert len(decision_ej9)>40
    print("Ejercicio 09: controles superados")
else:
    print("Ejercicio 09: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 10 · Errores de entrada y datos desconocidos

Completa `desconocido_ej10` evaluando una copia de P02 con calidad None. Define `capturo_error_ej10` como True si una p_excepcion=8 genera ValueError.

**Pista:** 8 no representa el ocho por ciento. El dato desconocido sí tiene una representación explícita.

In [ ]:
incompleta_ej10 = deepcopy(p02)
# TODO: calidad desconocida
desconocido_ej10 = None
capturo_error_ej10 = False
# TODO: intenta calcular una ficha con p_excepcion=8 y captura ValueError

In [ ]:
completado_ej10 = False
if completado_ej10:
    assert desconocido_ej10["puntuacion"] is None
    assert desconocido_ej10["estado"]=="pendiente"
    assert capturo_error_ej10 is True
    print("Ejercicio 10: controles superados")
else:
    print("Ejercicio 10: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 11 · Los pesos expresan preferencias

Construye `pesos_ej11` con impacto 0,50 y los demás factores 0,10. Evalúa y guarda los IDs de candidatos en `orden_ej11`. Explica en `lectura_ej11` qué cambia respecto al escenario base.

**Pista:** Recalcula los estados con la nueva puntuación, no solo reordenes el resultado anterior.

In [ ]:
pesos_ej11 = {}  # TODO
orden_ej11 = []  # TODO
lectura_ej11 = ""

In [ ]:
completado_ej11 = False
if completado_ej11:
    assert isclose(sum(pesos_ej11.values()),1)
    assert orden_ej11==["P02","P03","P01"]
    assert len(lectura_ej11)>40
    print("Ejercicio 11: controles superados")
else:
    print("Ejercicio 11: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Ejercicio 12 · Diseñar un piloto que pueda refutarse

Define `piloto_ej12` con hipotesis, alcance, medicion, aceptacion, detencion y responsable. Añade `limites_ej12` con al menos dos afirmaciones que el análisis no permite.

**Pista:** Debe ser posible observar una condición que obligue a cambiar la propuesta.

In [ ]:
piloto_ej12 = {"hipotesis":"", "alcance":"", "medicion":"", "aceptacion":"", "detencion":"", "responsable":""}
limites_ej12 = []

In [ ]:
completado_ej12 = False
if completado_ej12:
    assert set(piloto_ej12)=={"hipotesis","alcance","medicion","aceptacion","detencion","responsable"}
    assert all(isinstance(x,str) and x.strip() for x in piloto_ej12.values())
    assert len(limites_ej12)>=2
    print("Ejercicio 12: controles superados")
else:
    print("Ejercicio 12: pendiente; controles omitidos")

**Justifica tu respuesta:** explica la decisión y qué evidencia podría cambiarla.

## Entrega y rúbrica

Reinicia el kernel, ejecuta todo y entrega las doce tareas con sus banderas activadas. Evalúa cinco dimensiones, dos puntos cada una: alcance, coherencia de unidades, aplicación de filtros, contraste de escenarios y justificación del piloto.

## Fuentes

- Microsoft Learn, planificación: https://github.com/MicrosoftDocs/power-automate-docs/blob/main/articles/guidance/planning/introduction.md
- NHS England, mapeo: https://www.england.nhs.uk/improvement-hub/wp-content/uploads/sites/44/2017/11/ILG-1.2-Process-Mapping-Analysis-and-Redesign.pdf
- Python: https://docs.python.org/3/library/statistics.html y https://docs.python.org/3/library/math.html

El modelo numérico, las fichas, los pesos y los umbrales son originales y docentes. Las fuentes no certifican ni validan la puntuación. Referencias revisadas el 03/11/2026.